In [1]:
!pip install ultralytics
!pip install roboflow


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.8/302.8 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 139.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 6.9 MB/s eta 0:00:00
  Attempting uninstall: typer
    Found existing installation: typer 0.27.1
    Uninstalling typer-0.27.1:
      Successfully uninstalled typer-0.27.1


# get dataset

In [2]:
%pip install roboflow
import roboflow
print(roboflow.__version__)

1.4.2


In [ ]:

from roboflow import Roboflow



rf = Roboflow(api_key="ROBOFLOW_API_KEY")
project = rf.workspace("roboflow-jvuqo").project("football-players-detection-3zvbc")
version = project.version(1)
dataset = version.download("yolov5")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to football-players-detection-1 in yolov5pytorch:: 100%|██████████| 1338/1338 [00:00<00:00, 2947.92it/s]


In [4]:
dataset.location

'/content/football-players-detection-1'

In [5]:
import os

print(os.getcwd())
print(os.listdir("/content"))
print(os.listdir("/content/football-players-detection-1"))

/content
['.config', 'football-players-detection-1', 'sample_data']
['valid', 'test', 'data.yaml', 'README.dataset.txt', 'README.roboflow.txt', 'train']


In [6]:

import os

base = "/content/football-players-detection-1"

for root, dirs, files in os.walk(base):
    print(root)
    print("  folders:", dirs)
    print("  files:", files)

/content/football-players-detection-1
  folders: ['valid', 'test', 'train']
  files: ['data.yaml', 'README.dataset.txt', 'README.roboflow.txt']
/content/football-players-detection-1/valid
  folders: ['labels', 'images']
  files: []
/content/football-players-detection-1/valid/labels
  folders: []
  files: ['42ba34_1_4_png.rf.0190fbc779b7f4320c25835a5d952de0.txt', '744b27_7_6_png.rf.1bcd345f6691254eddeea6e9cb8425a4.txt', '40cd38_7_7_png.rf.734097f181067cdf149fbd63249a51b2.txt', '4b770a_5_5_png.rf.32e220f7f4f090859fd2bb2ba0c5d8d1.txt', '08fd33_3_3_png.rf.128b8280598b9931fdeeed42b5be4c51.txt', '08fd33_9_8_png.rf.cc61e7ba09940f4606e4464dd621fe2f.txt', '744b27_9_8_png.rf.68e5d3f2de2384cd05cdb3f5edbd051b.txt', '54745b_9_4_png.rf.3227ab0fa5e4b28148cbb235e5fa8494.txt', '573e61_1_10_png.rf.a01a3d439e964eb4342eb9e7d49295e6.txt', '538438_1_5_png.rf.927cf8101773ba75712dcee17a35e56b.txt', '798b45_1_1_png.rf.8439a1c181599a0556904a92fd89bd81.txt', '744b27_1_7_png.rf.b2d05a9295c0da84bb0f7107fa0b44f5.tx

# fix dataset paths


In [7]:
import os, yaml

root = dataset.location

def find_split_dir(root, split_names):
    """Walk the dataset folder and find the 'images' directory belonging to a given split,
    regardless of how deeply Roboflow nested it or whether it's named 'valid' or 'val'."""
    for dirpath, dirnames, filenames in os.walk(root):
        if os.path.basename(dirpath) == "images":
            parent = os.path.basename(os.path.dirname(dirpath))
            if parent in split_names:
                return dirpath
    return None

train_dir = find_split_dir(root, ["train"])
valid_dir = find_split_dir(root, ["valid", "val", "validation"])
test_dir  = find_split_dir(root, ["test"])

print("train:", train_dir)
print("valid:", valid_dir)
print("test: ", test_dir)

assert train_dir and valid_dir, "Could not locate train/valid image folders — check the folder tree above."

data_yaml_path = os.path.join(root, "data.yaml")

with open(data_yaml_path, "r") as f:
    data_cfg = yaml.safe_load(f)

# Use absolute paths directly and drop 'path', so nothing gets concatenated/doubled
data_cfg.pop("path", None)
data_cfg["train"] = train_dir
data_cfg["val"] = valid_dir
if test_dir:
    data_cfg["test"] = test_dir

with open(data_yaml_path, "w") as f:
    yaml.dump(data_cfg, f)

print("\nFixed data.yaml:")
print(yaml.dump(data_cfg))


train: /content/football-players-detection-1/train/images
valid: /content/football-players-detection-1/valid/images
test:  /content/football-players-detection-1/test/images

Fixed data.yaml:
names:
- ball
- goalkeeper
- player
- referee
nc: 4
roboflow:
  license: CC BY 4.0
  project: football-players-detection-3zvbc
  url: https://universe.roboflow.com/roboflow-jvuqo/football-players-detection-3zvbc/dataset/1
  version: 1
  workspace: roboflow-jvuqo
test: /content/football-players-detection-1/test/images
train: /content/football-players-detection-1/train/images
val: /content/football-players-detection-1/valid/images



# training

In [8]:
!yolo task=detect mode=train model=yolov5x.pt data={dataset.location}/data.yaml epochs=100 imgsz=640

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
PRO TIP 💡 Replace 'model=yolov5x.pt' with new 'model=yolov5xu.pt'.
YOLOv5 'u' models are trained with https://github.com/ultralytics/ultralytics and feature improved performance vs standard YOLOv5 models trained with https://github.com/ultralytics/yolov5.
Ultralytics 8.4.140 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/football-

**load the model to your drive**

In [10]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os

# change this to whatever folder name you want inside your Drive
drive_folder = '/content/drive/MyDrive/yolo_football_training'
os.makedirs(drive_folder, exist_ok=True)

weights_dir = '/content/runs/detect/train/weights'  # adjust if your run folder name differs (e.g. train-3)

shutil.copy(os.path.join(weights_dir, 'best.pt'), os.path.join(drive_folder, 'best.pt'))
shutil.copy(os.path.join(weights_dir, 'last.pt'), os.path.join(drive_folder, 'last.pt'))

print(f"Saved best.pt and last.pt to {drive_folder}")

Mounted at /content/drive
Saved best.pt and last.pt to /content/drive/MyDrive/yolo_football_training
